In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
sys.path.append('../../')
print(sys.path)


import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrames

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

# We only need the categories for this study and the expandable systs are really big
cols_to_keep = [
    ('nu_categ', '', '', ''),
    ('genie_categ', '', '', ''),
    ('genie_mode', '', '', ''),
    ('nu_categ_proton_reduced', '', '', ''),
    ('true_var','true_cos_theta_mu', '',''),
    ('true_var','true_cos_theta_pi', '',''),
    ('true_var','true_mu_pi_angle', '',''),
    ('true_var','true_p_mu', '',''),
    ('true_var','true_p_pi', '',''),
    ('true_var','num_protons', '',''),
    ('true_var','delta_pT', '',''),
    ('true_var','delta_alpha_T', '',''),
    ('true_var','delta_phi_T', '','')
]
mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load in time cosmic df
mc_in_time_cosmics_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_in_time_cosmics.df", keys2load, 100)
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_df['cc1pi']
mc_in_time_cosmics_hdr_df = mc_in_time_cosmics_df['hdr']

#Load CV lowE dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_lowE_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_lowE_CV.df", keys2load, 100)
mc_bnb_lowE_evt_df = mc_bnb_lowE_df['cc1pi']
mc_bnb_lowE_nu_df = mc_bnb_lowE_df['nudf']
mc_bnb_lowE_hdr_df = mc_bnb_lowE_df['hdr']
mc_bnb_lowE_nu_df = mc_bnb_lowE_nu_df[cols_to_keep]

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/old/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))


#Low E
mc_bnb_lowE_tot_pot = mc_bnb_lowE_hdr_df['pot'].sum()
print("dirt_tot_pot: %.3e" %(mc_bnb_lowE_tot_pot))
mc_bnb_lowE_pot_scale = data_tot_pot / mc_bnb_lowE_tot_pot
print("dirt_pot_scale: %.3e" %(mc_bnb_lowE_pot_scale))
mc_bnb_lowE_evt_df[pot_weight_col] = mc_bnb_lowE_pot_scale * np.ones(len(mc_bnb_lowE_evt_df))


intime_gates = mc_in_time_cosmics_hdr_df[mc_in_time_cosmics_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics data gates: {:.2e}".format(intime_gates))
f = 0.075
scale_intime_to_lightdata = (1-f)*data_gates/intime_gates
print("goal scale: {:.2f}".format(scale_intime_to_lightdata))
mc_in_time_cosmics_evt_df[pot_weight_col] = scale_intime_to_lightdata * np.ones(len(mc_in_time_cosmics_evt_df))

In [ ]:
mc_evt_df = mc_bnb_evt_df
if "ar23p" in bnb_path:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)   
print("Finished loading")    

mc_in_time_cosmics_evt_df = perform_truth_matching(mc_in_time_cosmics_evt_df, mc_bnb_nu_df.head(1))   
print("TM for cosmic done")

mc_evt_df = concat_shift_first_index(mc_evt_df,mc_in_time_cosmics_evt_df)
mc_bnb_lowE_evt_df = perform_truth_matching(mc_bnb_lowE_evt_df, mc_bnb_lowE_nu_df)
print("TM for BNB nu df")
mc_evt_df = concat_shift_first_index(mc_evt_df,mc_bnb_lowE_evt_df)

In [ ]:
if "ar23p" in bnb_path:
    new_columns = []
    for c in mc_bnb_nu_df.columns:
        new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
    mc_bnb_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)
    mc_bnb_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_nu_df))
else:
    mc_bnb_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_nu_df))

# Test background composition

In [ ]:
mc_evt_df =  mc_evt_df[build_event_cumulative_masks(mc_evt_df, sideband = "")["energy"]]

In [ ]:
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()
mc_evt_df = mc_evt_df[mc_evt_df.truth.nu_categ == "CC1pi"]

In [ ]:
HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))

In [ ]:
var_configs = [
    #VariableConfig.angle_between_candidates(),
    #VariableConfig.muon_momentum(),
    #VariableConfig.muon_direction(),
    #VariableConfig.pion_momentum(),
    #VariableConfig.pion_direction(),
    #VariableConfig.delta_phi_T(),
    VariableConfig.delta_alpha_T(),
    #VariableConfig.delta_pt(),
]

In [ ]:
import numpy as np
from tqdm import tqdm

# --- Configuration ---
n_attempts = 100000
n_bins = 3
min_entries_per_bin = 403.377*0.5/n_bins
EPSILON = 1e-6
completeness_threshold = 0.6
diagonal_check = 0.6

for var_config in var_configs:
    bin_edge_low = var_config.bins[0]
    bin_edge_high = var_config.bins[-1]
    reco_col = var_config.var_evt_reco_col
    true_col = var_config.var_evt_truth_col
    
    # Pre-extract data
    var_raw_reco = mc_evt_df[reco_col].values 
    var_raw_true = mc_evt_df[true_col].values 
    weights_raw = mc_evt_df[('slc', 'wgt', '', '', '', '')].values if ('slc', 'wgt', '', '', '', '') in mc_evt_df.columns else np.ones_like(var_raw_reco)

    # Clean NaNs
    mask = ~np.isnan(var_raw_reco) & ~np.isnan(var_raw_true)
    v_reco, v_true, w = var_raw_reco[mask], var_raw_true[mask], weights_raw[mask]

    # Target mean for "equal" bins
    total_counts = np.sum(w)
    target_mean = total_counts / n_bins

    possible_binnings = []
    
    for i in tqdm(range(n_attempts), desc=f"Searching {var_config.var_save_name}"):
        internal_edges = np.random.uniform(bin_edge_low, bin_edge_high, n_bins - 1)
        current_bins = np.sort(np.concatenate(([bin_edge_low], internal_edges, [bin_edge_high])))
    
        # 1. Prepare Histograms
        v_reco_clip = np.clip(v_reco, current_bins[0], current_bins[-1] - EPSILON)
        v_true_clip = np.clip(v_true, current_bins[0], current_bins[-1] - EPSILON)
        
        # reco_vs_true: Rows = Reco, Cols = True (standard convention)
        reco_vs_true, _, _ = np.histogram2d(v_true_clip, v_reco_clip, weights=w, bins=current_bins)
        nevts_reco, _ = np.histogram(v_reco_clip, weights=w, bins=current_bins)
        nevts_true, _ = np.histogram(v_true_clip, weights=w, bins=current_bins)
    
        # Condition 0: Minimum occupancy
        if not np.all(nevts_reco >= min_entries_per_bin):
            continue
    
        # --- CONDITION 1: Completeness (Stability) per bin ---
        diag_reco_true = np.diag(reco_vs_true)
        
        # Use np.errstate to handle division by zero safely
        with np.errstate(divide='ignore', invalid='ignore'):
            completeness = diag_reco_true / nevts_true
            completeness = np.nan_to_num(completeness) # Replace NaNs with 0
    
        if not np.all(completeness >= completeness_threshold): # 60% threshold
            continue
    
        # --- CONDITION 2: Global Diagonal Purity ---
        # (Sum of diagonal elements) / (Total events in the migration matrix)
        total_events_in_matrix = np.sum(reco_vs_true)
        trace_sum = np.trace(reco_vs_true)
        
        if total_events_in_matrix > 0:
            global_purity = trace_sum / total_events_in_matrix
        else:
            global_purity = completeness_threshold
    
        if global_purity < diagonal_check: # 68% threshold
            continue
        
            
        possible_binnings.append({
            "bins": current_bins,
            "counts": nevts_reco,
            "global_purity": global_purity,
            "completeness": completeness,
            "score": np.sum((nevts_reco - target_mean)**2) # Keep your uniformity score
        })
    
    
    # --- REPORTING ---
    print(f"\nSearch complete for {var_config.var_save_name}")
    print(f"Total attempts: {n_attempts} | Valid binnings found: {len(possible_binnings)}")
    
    if possible_binnings:
        # Sort candidates by score (best/lowest score first)
        possible_binnings.sort(key=lambda x: x["score"])
        #possible_binnings.sort(key=lambda x: (-x["global_purity"], x["score"]))
    
        # Header with updated column names
        # Adjusted width to 130 to accommodate extra data comfortably
        header = f"{'RANK':<8} | {'SCORE':<10} | {'GLOBAL PUR':<12} | {'MIN COMPL':<10} | {'BIN EDGES'}"
        print("\n" + "="*130)
        print(header)
        print("-" * 130)
        
        # Show top 20 candidates
        for idx, entry in enumerate(possible_binnings[:50]):
            # Formatting data
            b_str = ", ".join([f"{b:.3f}" for b in entry['bins']])
            g_pur = entry['global_purity'] * 100
            # Condition 1 was completeness >= 60%, so we show the minimum one found
            c_min = np.min(entry['completeness']) * 100
            
            prefix = "WINNER" if idx == 0 else f"#{idx+1}"
            
            # Main row: Rank, Score, Global Purity, Minimum Bin Completeness, and Edges
            print(f"{prefix:<8} | {entry['score']:<10.1f} | {g_pur:>10.1f}% | {c_min:>8.1f}% | [{b_str}]")
            
            # Sub-row: Show counts per bin for easy distribution checking
            counts_str = ", ".join([f"{c:.1f}" for c in entry['counts']])
            print(f"{'':<8} | Counts: [{counts_str}]")
            print("-" * 130)
    
        # Detailed summary of the chosen binning
        best = possible_binnings[0]
        print(f"\n>>> SELECTED BINNING FOR {var_config.var_save_name}:")
        print(f"    Edges:        {list(np.round(best['bins'], 4))}")
        print(f"    Global Pur:   {best['global_purity']*100:.2f}%")
        print(f"    Min Compl:    {np.min(best['completeness'])*100:.2f}%")
        print(f"    Total Score:  {best['score']:.4f}")
        
    else:
        print(f"\n[!] No valid binnings found for {var_config.var_save_name}.")
        print("    Consider: Increasing n_attempts, lowering completeness_threshold, or lowering min_entries_per_bin.")